# Section 12: Conjugates of Convex Functions

**Source span.** Rockafellar, *Convex Analysis*, printed pages 102-111 (PDF pages 120-129). The PDF is used here only for structure and terminology. The prose, examples, diagrams, and checks in this notebook are original synthetic course material with no copied text, screenshots, crops, or textbook figures.

**Section goal.** Build an inspectable model of Fenchel conjugacy, affine minorants, biconjugacy, dual slopes. By the end, the key objects should feel less like formal definitions and more like things you can test: a point either satisfies the affine or convex certificate, an epigraph boundary either closes correctly, a separating functional either has a positive margin, or a dual value either matches its primal partner.


In [ ]:
# geometry-setup:v1
# Machine-managed by scripts/update_notebook_setup.py. Do not edit this cell by hand.

from __future__ import annotations

import json as _geometry_json
import os as _geometry_os
from pathlib import Path as _GeometryPath
import sys as _geometry_sys

GEOMETRY_SETUP = _geometry_json.loads(
    r"""
{
  "colab_url": "https://colab.research.google.com/github/Rah-Rah-Mitra/Geometry/blob/main/Convex-Analysis/part-03-duality-correspondences/section-12-conjugates-of-convex-functions/12-conjugates-of-convex-functions.ipynb",
  "course_dir": "Convex-Analysis",
  "course_title": "Convex Analysis",
  "github_url": "https://github.com/Rah-Rah-Mitra/Geometry/blob/main/Convex-Analysis/part-03-duality-correspondences/section-12-conjugates-of-convex-functions/12-conjugates-of-convex-functions.ipynb",
  "jupyterlite": false,
  "marker": "geometry-setup:v1",
  "notebook_kind": "lesson",
  "notebook_path": "Convex-Analysis/part-03-duality-correspondences/section-12-conjugates-of-convex-functions/12-conjugates-of-convex-functions.ipynb",
  "notebook_title": "Section 12: Conjugates of Convex Functions",
  "repository": {
    "branch": "main",
    "name": "Geometry",
    "owner": "Rah-Rah-Mitra",
    "source_url": "https://github.com/Rah-Rah-Mitra/Geometry"
  },
  "requirements": "requirements/algebraic-geometry.txt",
  "runtime_profile": "algebraic_geometry"
}
"""
)


def _geometry_is_colab():
    try:
        import google.colab  # type: ignore  # noqa: F401
        return True
    except Exception:
        return False


def _geometry_is_jupyterlite():
    return _geometry_sys.platform == "emscripten" or "pyodide" in _geometry_sys.modules


def _geometry_add_path(path):
    text = str(path)
    if text not in _geometry_sys.path:
        _geometry_sys.path.insert(0, text)


def _geometry_find_repo_root():
    candidates = []
    env_root = _geometry_os.environ.get("GEOMETRY_REPO_ROOT")
    if env_root:
        candidates.append(_GeometryPath(env_root).expanduser())
    candidates.append(_GeometryPath.cwd())
    for start in candidates:
        start = start.resolve()
        for current in (start, *start.parents):
            if (current / "course-manifest.json").exists() and (
                current / "metadata" / "runtime_profiles.yml"
            ).exists():
                return current
    raise RuntimeError(
        "Could not find the Geometry repository root. Start JupyterLab inside the "
        "Geometry checkout or set GEOMETRY_REPO_ROOT."
    )


def _geometry_run(command):
    import subprocess as _geometry_subprocess

    printable = " ".join(str(part) for part in command)
    print(f"+ {printable}")
    _geometry_subprocess.check_call([str(part) for part in command])


def _geometry_requirement_names(requirements_path, seen=None):
    seen = set() if seen is None else seen
    requirements_path = requirements_path.resolve()
    if requirements_path in seen or not requirements_path.exists():
        return []
    seen.add(requirements_path)
    names = []
    for raw_line in requirements_path.read_text(encoding="utf-8").splitlines():
        line = raw_line.split("#", 1)[0].strip()
        if not line:
            continue
        if line.startswith(("-r ", "--requirement ")):
            _, nested = line.split(maxsplit=1)
            names.extend(_geometry_requirement_names(requirements_path.parent / nested, seen))
            continue
        if line.startswith("-"):
            continue
        name = line
        for separator in ("==", ">=", "<=", "~=", "!=", ">", "<", ";"):
            name = name.split(separator, 1)[0]
        name = name.split("[", 1)[0].strip()
        if name:
            names.append(name)
    return sorted(set(names))


def _geometry_missing_requirements(requirements_path):
    import importlib.metadata as _geometry_metadata

    missing = []
    for name in _geometry_requirement_names(requirements_path):
        try:
            _geometry_metadata.distribution(name)
        except _geometry_metadata.PackageNotFoundError:
            missing.append(name)
    return missing


def _geometry_configured_roots(repo_root):
    course_dir = GEOMETRY_SETUP.get("course_dir")
    course_root = repo_root / course_dir if course_dir else repo_root
    return repo_root, course_root


if _geometry_is_jupyterlite():
    if not GEOMETRY_SETUP["jupyterlite"]:
        raise RuntimeError(
            "This Geometry notebook uses runtime profile "
            f"{GEOMETRY_SETUP['runtime_profile']!r}, which is not enabled for "
            "JupyterLite in course-manifest.json. Open it in Colab or local JupyterLab."
        )
    GEOMETRY_REPO_ROOT = _GeometryPath.cwd()
    GEOMETRY_COURSE_ROOT = (
        GEOMETRY_REPO_ROOT / GEOMETRY_SETUP["course_dir"]
        if GEOMETRY_SETUP.get("course_dir")
        else GEOMETRY_REPO_ROOT
    )
    _geometry_add_path(GEOMETRY_REPO_ROOT)
    if GEOMETRY_COURSE_ROOT.exists():
        _geometry_add_path(GEOMETRY_COURSE_ROOT)
    GEOMETRY_RUNTIME_PROFILE = GEOMETRY_SETUP["runtime_profile"]
    print(
        "Geometry setup: JupyterLite/Pyodide detected; shell, git, and pip steps "
        "were skipped."
    )
elif _geometry_is_colab():
    repository = GEOMETRY_SETUP["repository"]
    repo_url = repository["source_url"].rstrip("/") + ".git"
    branch = repository["branch"]
    GEOMETRY_REPO_ROOT = _GeometryPath(
        _geometry_os.environ.get("GEOMETRY_REPO_ROOT", "/content/Geometry")
    )
    sparse_paths = ["requirements", "metadata", "scripts", "course-manifest.json", "index.ipynb"]
    if GEOMETRY_SETUP.get("course_dir"):
        sparse_paths.append(GEOMETRY_SETUP["course_dir"])
    if not (GEOMETRY_REPO_ROOT / ".git").exists():
        if GEOMETRY_REPO_ROOT.exists() and any(GEOMETRY_REPO_ROOT.iterdir()):
            raise RuntimeError(
                f"{GEOMETRY_REPO_ROOT} exists but is not a git checkout. "
                "Set GEOMETRY_REPO_ROOT to an empty path or remove the directory."
            )
        _geometry_run(
            [
                "git",
                "clone",
                "--filter=blob:none",
                "--no-checkout",
                "--branch",
                branch,
                repo_url,
                GEOMETRY_REPO_ROOT,
            ]
        )
        _geometry_run(["git", "-C", GEOMETRY_REPO_ROOT, "sparse-checkout", "init", "--cone"])
    _geometry_run(["git", "-C", GEOMETRY_REPO_ROOT, "sparse-checkout", "set", *sparse_paths])
    _geometry_run(["git", "-C", GEOMETRY_REPO_ROOT, "checkout", branch])
    requirements_path = GEOMETRY_REPO_ROOT / GEOMETRY_SETUP["requirements"]
    _geometry_run([_geometry_sys.executable, "-m", "pip", "install", "-q", "-r", requirements_path])
    GEOMETRY_REPO_ROOT, GEOMETRY_COURSE_ROOT = _geometry_configured_roots(GEOMETRY_REPO_ROOT)
    _geometry_os.chdir(GEOMETRY_COURSE_ROOT if GEOMETRY_COURSE_ROOT.exists() else GEOMETRY_REPO_ROOT)
    _geometry_add_path(GEOMETRY_REPO_ROOT)
    _geometry_add_path(GEOMETRY_COURSE_ROOT)
    GEOMETRY_RUNTIME_PROFILE = GEOMETRY_SETUP["runtime_profile"]
    print(
        f"Geometry setup: Colab ready at {_GeometryPath.cwd()} "
        f"with profile {GEOMETRY_RUNTIME_PROFILE!r}."
    )
else:
    GEOMETRY_REPO_ROOT = _geometry_find_repo_root()
    requirements_path = GEOMETRY_REPO_ROOT / GEOMETRY_SETUP["requirements"]
    missing = _geometry_missing_requirements(requirements_path)
    skip_install = _geometry_os.environ.get("GEOMETRY_SKIP_INSTALL") == "1"
    if missing and skip_install:
        print(
            "Geometry setup: GEOMETRY_SKIP_INSTALL=1, so missing profile packages "
            f"were not installed: {', '.join(missing)}"
        )
    elif missing:
        print(
            "Geometry setup: installing missing profile packages from "
            f"{requirements_path.relative_to(GEOMETRY_REPO_ROOT)}: {', '.join(missing)}"
        )
        _geometry_run([_geometry_sys.executable, "-m", "pip", "install", "-r", requirements_path])
    GEOMETRY_REPO_ROOT, GEOMETRY_COURSE_ROOT = _geometry_configured_roots(GEOMETRY_REPO_ROOT)
    _geometry_os.chdir(GEOMETRY_COURSE_ROOT if GEOMETRY_COURSE_ROOT.exists() else GEOMETRY_REPO_ROOT)
    _geometry_add_path(GEOMETRY_REPO_ROOT)
    _geometry_add_path(GEOMETRY_COURSE_ROOT)
    GEOMETRY_RUNTIME_PROFILE = GEOMETRY_SETUP["runtime_profile"]
    print(
        f"Geometry setup: local checkout ready at {_GeometryPath.cwd()} "
        f"with profile {GEOMETRY_RUNTIME_PROFILE!r}."
    )


## Translation Guide

Part III is the duality spine of the course. Each section is treated as a way of replacing a set or function by the hyperplanes, slopes, or polar objects that certify it.

For this section, the translation into computational language is centered on Fenchel conjugate, affine minorant, biconjugate, duality pairing. The notebook treats those ideas as objects with coordinates, inequalities, sampled witnesses, and explicit residuals. That is deliberately narrower than the full theorem set of the printed section, but it is broad enough to make the main geometry visible and to support later refinement by a section worker.

- `Fenchel conjugate`: the slope-indexed supremum pairing a function with affine minorants.
- `affine minorant`: an affine function lying below a convex function.
- `biconjugate`: the conjugate of the conjugate, recovering closed convex functions.
- `duality pairing`: the dot product term that couples primal points and dual slopes.


## Library Routing

- `numpy` supplies the finite-dimensional examples, sampled points, residuals, and small matrix calculations.
- `matplotlib` creates durable static diagrams under `artifacts/part-03-duality-correspondences/section-12-conjugates-of-convex-functions/figures`.
- `networkx` records the proof or concept dependency map, so the logical route is visible rather than buried in prose.
- `sympy` is available for exact symbolic checks in sections where conjugacy, saddle reconstruction, or algebra identities benefit from exact expressions.

The main visual plan is: A quadratic is reconstructed from its supporting affine minorants indexed by slope. The associated invariant is: Fenchel equality holds at the slope paired with the sampled primal point.


In [ ]:
from pathlib import Path
import sys

import pandas as pd


def find_book_root(start=Path.cwd()):
    current = Path(start).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "AGENTS.md").exists() and (candidate / "Convex Analysis.pdf").exists():
            return candidate
    raise RuntimeError("Could not find Convex Analysis course root")


BOOK_ROOT = find_book_root()
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

from utils.artifacts import assert_artifacts, display_artifact, relative_to_book
from utils.convex_examples import create_section_artifacts
from utils.section_catalog import get_section

section = get_section(12)
artifact_dir = BOOK_ROOT / "artifacts" / section["artifact_key"]
section_summary = {
    "section": section["number"],
    "title": section["title"],
    "printed_pages": f"{section['printed_start']}-{section['printed_end']}",
    "pdf_pages": f"{section['pdf_start']}-{section['pdf_end']}",
    "artifact_dir": relative_to_book(artifact_dir, BOOK_ROOT),
}
section_summary


## Visual Construction

The first artifact is the section's concrete geometry lab. Read it as a test bench, not as decoration. The learner should inspect this target: A quadratic is reconstructed from its supporting affine minorants indexed by slope. In Rockafellar's style, many proofs move by replacing an object with a certificate. Here the certificate is visible in the plotted point, line, cone, epigraph, support direction, multiplier, or saddle field.


In [ ]:
artifact_bundle = create_section_artifacts(section, artifact_dir)
[relative_to_book(path, BOOK_ROOT) for path in artifact_bundle["paths"]]


In [ ]:
for artifact in artifact_bundle["display"]:
    display_artifact(artifact)


## Inspection Table

The table makes the visual contract explicit. Each row connects a section concept to the same inspection target and invariant, so that a future worker can deepen the notebook without losing the original source-map alignment. The dependency map follows this route:

`affine minorants -> slope parameter -> conjugate -> Fenchel equality`


In [ ]:
inspection_table = pd.DataFrame(artifact_bundle["inspection_rows"])
inspection_table


## What To Check In The Figure

Start with the artifact before reading the theorem statements in the PDF. Ask what would break if the plotted witness moved, if a boundary value were missing, if the normal changed sign, or if the dual pairing failed. The check below is intentionally small: Fenchel equality holds at the slope paired with the sampled primal point. A small residual does not prove every theorem in the section, but it does keep the central invariant honest and executable.

For a teaching pass, this is the place to add more examples: a degenerate case, a boundary case, and a failed hypothesis. For this course scaffold, the notebook gives one reliable model that later workers can enrich without changing folder structure.


In [ ]:
checks = artifact_bundle["checks"]
checks


## Learner Lab

Modify one ingredient in the construction and rerun the artifact cell. Good experiments for this section are tied to the concepts `Fenchel conjugate, affine minorant, biconjugate, duality pairing`. Keep the invariant in view: if the object is convex, average two displayed feasible points; if the object is dual, test the pairing; if the object is a subgradient or multiplier, test the supporting inequality or stationarity residual; if the object is saddle-shaped, test the two optimization directions separately.

A useful lab report has three entries: the changed parameter, the visual consequence, and whether the JSON check still passes. When it fails, the failure is not noise; it is the geometry telling you which hypothesis was doing work.


In [ ]:
source_span_ok = (
    section["printed_start"] == 102
    and section["printed_end"] == 111
    and section["pdf_start"] == 120
    and section["pdf_end"] == 129
)

final_sanity = {
    "source_span_ok": source_span_ok,
    "artifact_count": len(artifact_bundle["paths"]),
    "primary_invariant_ok": bool(checks["primary_invariant_ok"]),
    "artifact_dir": relative_to_book(artifact_dir, BOOK_ROOT),
}
assert source_span_ok
assert checks["primary_invariant_ok"]
assert_artifacts(artifact_bundle["paths"], min_bytes=64)
final_sanity


## Takeaways

- The section's core geometry is made visible through A quadratic is reconstructed from its supporting affine minorants indexed by slope.
- The executable invariant is: Fenchel equality holds at the slope paired with the sampled primal point.
- The source map points to printed pages 102-111 and PDF pages 120-129, so a later author can deepen the lesson while keeping the canonical section boundary.
- The visual is synthetic and original; it is meant to teach the idea, not reproduce the book's layout or figures.
